In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
!pip install jenkspy

In [ ]:
!pip install scikit-posthocs

In [ ]:
df = pd.read_csv("data_cleaning_2026_27_04.csv")

In [ ]:
df.columns

In [ ]:
df_clean = df.loc[:, ~df.columns.isin(['mascota', 'peso', 'estatura', 'bebedor_social', 'fumador_social'])]

In [ ]:
df_clean = df_clean[df_clean['horas_ausentismo'] > 0]
df_clean.info()

Pregunta: "tener hijos incide sul tiempo de ausencia?" - Aplico el test de Mann-Whitney U 

In [ ]:
from scipy import stats

con_hijos = df[df['hijos'] > 0]['horas_ausentismo']
sin_hijos = df[df['hijos'] == 0]['horas_ausentismo']

stat, p = stats.mannwhitneyu(con_hijos, sin_hijos, alternative='two-sided')
print(f"p-valor: {p:.4f}")
print("Diferencia significativa" if p < 0.05 else "Sin diferencia significativa")

print(f"\nMediana con hijos:  {con_hijos.median():.1f} h")
print(f"Mediana sin hijos:  {sin_hijos.median():.1f} h")

El rango de edad: el metodo para encontrar los cortes donde la variación dentro del grupo es mínima y entre grupos es máxima. Diseñado para agrupar una variable continua de forma objetiva.

In [ ]:
import jenkspy
import scikit_posthocs as sp
from scipy import stats

# 1. Crear grupos de edad con Jenks Natural Breaks
breaks = jenkspy.jenks_breaks(df_clean['edad'], n_classes=4)
df_clean['rango_edad'] = pd.cut(df_clean['edad'], bins=breaks, include_lowest=True)

print("Distribución de grupos:")
print(df_clean['rango_edad'].value_counts().sort_index())

# 2. Kruskal-Wallis
grupos = [g['horas_ausentismo'].values
          for _, g in df_clean.groupby('rango_edad', observed=True)]

stat, p = stats.kruskal(*grupos)
print(f"\nKruskal-Wallis — p-valor: {p:.4f}")

# 3. Post-hoc Dunn solo si hay diferencia significativa
if p < 0.05:
    print("\nDiferencia significativa — test de Dunn (Bonferroni):")
    print(sp.posthoc_dunn(df_clean, val_col='horas_ausentismo',
                          group_col='rango_edad', p_adjust='bonferroni'))
else:
    print("Sin diferencia significativa entre grupos de edad.")